# M1 walking skeleton on Colab (A100)

Runs the whole M1 experiment with one command: **4 baseline configs × 20 BFCL multi-turn tasks × 2 seeds + R1/R2/R3 × 20 tasks × 1 seed = 220 episodes**, then writes `report.md`.

* Runtime: **A100** (Runtime → Change runtime type). High-RAM helps the CPU-backend config.
* Everything resumable: if Colab disconnects, re-run all cells; finished episodes are skipped (results live on Drive).
* Results are **development data** (M1), excluded from the study.

| Step | Cell | Rough time on A100 |
|---|---|---|
| Setup (first time builds llama.cpp, then cached on Drive) | 1–3 | 15 min first, ~2 min after |
| Model files + R2 Q8_0 | 4 | 2 min |
| R3: toy QAD (S7) + export + checks | 5 | 10 min |
| Dry run, then the run | 6–7 | GPU configs ~1 h; the CPU config (40 episodes, prompt cache off) is the long pole: several hours, resumable |
| Report | 8 | < 1 min |

In [ ]:
# 1. Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive'
PILOT_MODELS = f'{DRIVE}/CMPE295A_pilot_2026-10-08/models'   # BF16, Q4_K_M, Q4_0-pure from the Oct 8 pilot
CACHE = f'{DRIVE}/CMPE295A_m1_cache'                          # llama.cpp builds, R3 outputs, uv cache
OUT = f'{DRIVE}/CMPE295A_m1_runs'                             # experiment results (resumable)
BRANCH = 'm1/skeleton'
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
os.environ.update(TESTBED_OUT=OUT, TESTBED_MODELS='/content/models',
                  LLAMA_CUDA_BIN='/content/cmpe_295a/llama.cpp/build-cuda/bin/llama-server',
                  LLAMA_CPU_BIN='/content/cmpe_295a/llama.cpp/build-cpu/bin/llama-server',
                  UV_CACHE_DIR=f'{CACHE}/uv')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc

In [ ]:
# 2. Code (private repo: add a GitHub token as the Colab secret GH_TOKEN)
import subprocess
try:
    from google.colab import userdata; tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/VritikaMalhotra/cmpe_295a' if tok else 'https://github.com/VritikaMalhotra/cmpe_295a'
if not os.path.exists('/content/cmpe_295a'):
    subprocess.run(['git', 'clone', '-b', BRANCH, url, '/content/cmpe_295a'], check=True)
else:
    subprocess.run(['git', '-C', '/content/cmpe_295a', 'pull', '--ff-only'], check=True)
%cd /content/cmpe_295a
!git log -1 --oneline

In [ ]:
# 3. Environment + llama.cpp (pinned to the pilot's commit; build cached on Drive)
!bash scripts/setup_env.sh --llama --cuda --cache {CACHE}

In [ ]:
# 4. Model files: pilot GGUFs from Drive (SHA-256 checked) + R2's Q8_0 built from the same BF16
!mkdir -p /content/models && cp -n {PILOT_MODELS}/Qwen3-0.6B-BF16.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_K_M.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_0-pure.gguf /content/models/
!cp -n {CACHE}/Qwen3-0.6B-Q8_0.gguf /content/models/ 2>/dev/null || true
!bash scripts/make_variants.sh /content/models llama.cpp/build-cuda/bin/llama-quantize
!cp -n /content/models/Qwen3-0.6B-Q8_0.gguf {CACHE}/

In [ ]:
# 5. R3: rerun the S7 toy QAD (50 steps; the Oct 8 checkpoint was not kept) with the llama.cpp-exact
#    Q4_0 quantizer (train/qad_toy.py), export to GGUF, check weight fidelity + KL
R3 = f'{CACHE}/r3'
if not os.path.exists(f'{R3}/Qwen3-0.6B-S7QAD-Q4_0-pure.gguf'):
    !uv pip install -q --python .venv/bin/python -e ".[train]" datasets
    !.venv/bin/hf download Qwen/Qwen3-0.6B --local-dir /content/work/Qwen3-0.6B
    !mkdir -p /content/work/s7 && .venv/bin/python train/make_qad_data.py --tokenizer /content/work/Qwen3-0.6B --out /content/work/s7/seqs.pt
    !cd /content/work/s7 && /content/cmpe_295a/.venv/bin/python /content/cmpe_295a/train/qad_toy.py --model /content/work/Qwen3-0.6B --out runA --data seqs.pt | tail -3
    !.venv/bin/python train/export_qad.py --base /content/work/Qwen3-0.6B --ckpt /content/work/s7/runA/ckpt.pt --out {R3} --llama llama.cpp --data /content/work/s7/seqs.pt
    !cp /content/work/s7/seqs.json /content/work/s7/runA/loss.jsonl {R3}/
!cp -n {R3}/Qwen3-0.6B-S7QAD-Q4_0-pure.gguf /content/models/
!cat {R3}/export_report.json | head -40

In [ ]:
# 6. Dry run: what will run, what is already done, what is missing
!.venv/bin/python -m testbed.run +experiment=skeleton dry_run=true

In [ ]:
# 7. Run (re-run this cell after a disconnect; it resumes)
!.venv/bin/python -m testbed.run +experiment=skeleton local=true 2>&1 | tee -a {OUT}/colab_run.log | grep -v '^  multi_turn' | tail -40

In [ ]:
# 8. Report
import glob
d = sorted(glob.glob(f'{OUT}/skeleton/*/'), key=os.path.getmtime)[-1]
!.venv/bin/python -m testbed.run +experiment=skeleton report_only=true report_partial=true
from IPython.display import Markdown, display
display(Markdown(open(d + 'report.md').read()))